In [1]:
# from preprocessing import get_model_dataset, create_train_test, min_max_scale, df_to_xy, read_file, lag_features
from lstm import create_model
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from datetime import date
from pathlib import Path

In [2]:
def read_file(file):
    """Read a single file and return a dataframe"""
    return pd.read_csv(file, skipinitialspace=True)

def lag_features(df, features, seq_length):
    """Transforms a raw 2D dataframe of option data into 2D dataframe ofsequence data.
    Last 2 indexes per sequence are bid and ask price. The len(features)*seq_length
    features before are sequences of features"""
    df = df.sort_values(["Expire_date", "Strike", "Ttl"], ascending = [True, True, False])
    
    # Adding lag for naive benchmarking
    #df["Naive"] = df["Price"].shift(1)

    for step in range(seq_length)[::-1]:
        for feature in features:
            df[feature + "-" + str(step)] = df[feature].shift(step)
    
    df["Check_strike"] = df["Strike"] == df["Strike"].shift(seq_length-1)
    df["Check_expire"] = df["Expire_date"] == df["Expire_date"].shift(seq_length-1)
    df = df[(df["Check_strike"] == True) & (df["Check_expire"] == True)]
    df = df.drop(["Check_strike", "Check_expire"], axis=1)
    #df[["Bid_strike_last", "Ask_strike_last"]] = df[["Bid_strike", "Ask_strike"]]
    #df[["Bid_last", "Ask_last"]] = df[["Bid", "Ask"]]
    df["Price_last"] = df["Price"]
    df = df.sort_values(["Quote_date"], ascending = [True])
    return df

def df_to_xy(df, num_features, seq_length, num_outputs):
    """Transforms a dataframe into two arrays of explanatory variables x and explained variables y"""
    array = df.to_numpy()
    array_x, array_y = array[:, -num_features*seq_length - num_outputs:-num_outputs].astype(np.float32), array[:,-num_outputs:].astype(np.float32)
    return array_x, array_y

In [3]:
first_year = 2019
last_year = 2021
file = f"../data/processed_data/{first_year}-{last_year}_underlying-strike_only-price.csv"

df_read = read_file(file)
print(df_read)
df_read.info()
print(df_read)
print(df_read["Ttl"].max())

         Unnamed: 0  Quote_date Expire_date     Price  Underlying_last  \
0           1354913  2019-01-02  2019-01-04  1707.050          2509.98   
1           1354914  2019-01-02  2019-01-04  1607.495          2509.98   
2           1354915  2019-01-02  2019-01-04  1507.500          2509.98   
3           1354916  2019-01-02  2019-01-04  1458.295          2509.98   
4           1354917  2019-01-02  2019-01-04  1408.300          2509.98   
...             ...         ...         ...       ...              ...   
5123793     6521988  2021-12-31  2024-12-20   150.000          4766.39   
5123794     6521989  2021-12-31  2024-12-20   150.000          4766.39   
5123795     6521990  2021-12-31  2024-12-20   150.900          4766.39   
5123796     6521991  2021-12-31  2024-12-20   150.000          4766.39   
5123797     6521992  2021-12-31  2024-12-20   150.000          4766.39   

         Strike   Ttl  Volatility  Volatility_GJR_GARCH     R  
0         800.0     2    0.202726              

In [4]:
from datetime import datetime
from dateutil.relativedelta import relativedelta

training_period = 10
val_period = 1
test_period = 1
num_models = 12

features = ["Underlying_last", "Strike", "Ttl", "Volatility_GJR_GARCH", "R"]
seq_length = 5
num_features = len(features)
num_outputs = 1

df_read_lags = lag_features(df_read, features, seq_length)

train_val_test = []

month = 4
year = 0
for i in range(num_models):
    if month == 13:
        year += 1
        month = 1
    train_start = datetime(2020 + year, month, 1)
    val_start = train_start + relativedelta(months=8)
    test_start = val_start + relativedelta(months=1)
    test_end = test_start + relativedelta(months=1)

    month += 1

    df_train_orginal = df_read_lags.loc[(df_read_lags.loc[:, "Quote_date"] >= str(train_start)) & (df_read_lags.loc[:, "Quote_date"] < str(val_start)), :]
    df_val_orginal = df_read_lags.loc[(df_read_lags.loc[:, "Quote_date"] >= str(val_start)) & (df_read_lags.loc[:, "Quote_date"] < str(test_start)), :]
    df_test_orginal = df_read_lags.loc[(df_read_lags.loc[:, "Quote_date"] >= str(test_start)) & (df_read_lags.loc[:, "Quote_date"] < str(test_end)), :]

    train_x_org, train_y_org = df_to_xy(df_train_orginal, num_features, seq_length, num_outputs)
    val_x_org, val_y_org = df_to_xy(df_val_orginal, num_features, seq_length, num_outputs)
    test_x_org, test_y_org = df_to_xy(df_test_orginal, num_features, seq_length, num_outputs)

    scaler = MinMaxScaler()
    train_x_scaled = scaler.fit_transform(train_x_org)
    val_x_scaled = scaler.transform(val_x_org)
    test_x_scaled = scaler.transform(test_x_org)

    print(month, test_x_scaled.shape)
    print(test_start, test_end)

    """shuffle = np.random.permutation(len(train_x_scaled))
    train_x_scaled, train_y_scaled = train_x_scaled[shuffle], train_y_scaled[shuffle]"""

    train_x_scaled = np.reshape(train_x_scaled, (len(train_x_scaled), seq_length, num_features))
    val_x_scaled = np.reshape(val_x_scaled, (len(val_x_scaled), seq_length, num_features))
    test_x_scaled = np.reshape(test_x_scaled, (len(test_x_scaled), seq_length, num_features))

    # print(f"Train_x shape: {train_x_scaled.shape}, train_y shape: {train_y_org.shape}")
    # print(f"Test_x shape: {test_x_scaled.shape}, test_y shape: {test_y_org.shape}")
    # print("------------------------------------------------")
    train_val_test.append(((train_x_scaled, train_y_org), (val_x_scaled, val_y_org), (test_x_scaled, test_y_org)))




5 (134329, 25)
2021-01-01 00:00:00 2021-02-01 00:00:00
6 (123140, 25)
2021-02-01 00:00:00 2021-03-01 00:00:00
7 (159777, 25)
2021-03-01 00:00:00 2021-04-01 00:00:00
8 (149256, 25)
2021-04-01 00:00:00 2021-05-01 00:00:00
9 (155258, 25)
2021-05-01 00:00:00 2021-06-01 00:00:00
10 (152434, 25)
2021-06-01 00:00:00 2021-07-01 00:00:00
11 (153033, 25)
2021-07-01 00:00:00 2021-08-01 00:00:00
12 (169230, 25)
2021-08-01 00:00:00 2021-09-01 00:00:00
13 (142146, 25)
2021-09-01 00:00:00 2021-10-01 00:00:00
2 (162490, 25)
2021-10-01 00:00:00 2021-11-01 00:00:00
3 (168877, 25)
2021-11-01 00:00:00 2021-12-01 00:00:00
4 (175512, 25)
2021-12-01 00:00:00 2022-01-01 00:00:00


In [5]:
# df_to = df_test_orginal

# df_a = df_to[(df_to["Expire_date"] == "2021-05-19") & (df_to["Strike"] == 3500)]

# print(df_a)

In [6]:
# df_ax, df_ay = df_to_xy(df_a, num_features, seq_length, num_outputs)

# print(df_ax[-1])
# print(df_ay[-1])

In [15]:
from keras.callbacks import EarlyStopping
import tensorflow as tf

config = {
        "units": 32,
        "learning_rate": 0.0012930813401443829,
        "layers": 4,
        "bn_momentum" : 0.001873745946822547,
        "weight_decay": 0.00005187188064474824,
        "seq_length": seq_length,
        "num_features": num_features,
    }

def trainer(train_x, train_y, model, val_x, val_y):
    epochs = 100
    minibatch_size = 4096

    tf.random.set_seed(2)

    early_stopping = EarlyStopping(
        monitor='val_loss',
        mode='min',
        min_delta = 1,
        patience = 5,
    )

    model.fit(
        train_x,
        train_y,
        batch_size = minibatch_size,
        # validation_split = 0.3,
        validation_data = (val_x, val_y),
        epochs = epochs,
        callbacks = [early_stopping]
    )

predictions = []
for i, ((x_train, y_train), (x_val, y_val), (x_test, y_test)) in enumerate(train_val_test):
    if i == 11:
        model = create_model(config)
        model.summary()
        trainer(x_train, y_train, model, x_val, y_val)
        predictions.append(np.array(model(x_test)))

# predictions = np.array(predictions)
predictions = np.concatenate(predictions)


"""path = f"./runs/model_w_validation/{first_year}-{last_year}-{date.today()}"
model.save(path)"""

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:204: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_15"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_60 (LSTM)                  │ (None, 5, 32)          │         4,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_60          │ (None, 5, 32)          │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_61 (LSTM)                  │ (None, 5, 32)          │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_61          │ (None, 5, 32)          │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_62 (LSTM)                  │ (None, 5, 32)          │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_62          │ (None, 5, 32)          │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_63 (LSTM)                  │ (None, 32)             │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_63          │ (None, 32)             │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 30,369 (118.63 KB)

 Trainable params: 30,113 (117.63 KB)

 Non-trainable params: 256 (1.00 KB)

Epoch 1/100
304/304 ━━━━━━━━━━━━━━━━━━━━ 24s 68ms/step - loss: 1136875.6250 - mae: 695.0871 - val_loss: 1249821.5000 - val_mae: 731.0163
Epoch 2/100
304/304 ━━━━━━━━━━━━━━━━━━━━ 21s 68ms/step - loss: 1062511.5000 - mae: 675.0149 - val_loss: 1117045.5000 - val_mae: 692.7499
Epoch 3/100
304/304 ━━━━━━━━━━━━━━━━━━━━ 19s 63ms/step - loss: 937250.8125 - mae: 636.5748 - val_loss: 939513.7500 - val_mae: 638.0453
Epoch 4/100
304/304 ━━━━━━━━━━━━━━━━━━━━ 19s 63ms/step - loss: 773855.5000 - mae: 580.4537 - val_loss: 730880.1250 - val_mae: 564.2543
Epoch 5/100
304/304 ━━━━━━━━━━━━━━━━━━━━ 80s 264ms/step - loss: 595211.2500 - mae: 510.4245 - val_loss: 530211.6875 - val_mae: 482.9995
Epoch 6/100
304/304 ━━━━━━━━━━━━━━━━━━━━ 23s 75ms/step - loss: 423170.8125 - mae: 431.1560 - val_loss: 354839.0000 - val_mae: 393.9833
Epoch 7/100
304/304 ━━━━━━━━━━━━━━━━━━━━ 23s 74ms/step - loss: 274712.8750 - mae: 347.6086 - val_loss: 214951.3594 - val_mae: 307.8183
Epoch 8/100
304/304 ━━━━━━━━━━━━━━━━━━━━ 21s 70ms/

'path = f"./runs/model_w_validation/{first_year}-{last_year}-{date.today()}"\nmodel.save(path)'

In [8]:
def prediction(df_test, predictions):
    # df_test["Prediction"] = predictions.flatten()
    df_test["Prediction"] = predictions
    return df_test

df_test_whole = df_read_lags.loc[df_read_lags.loc[:, "Quote_date"] >= "2021-01-01", :]
df_test_whole = prediction(df_test_whole, predictions)

from datetime import datetime
time = datetime.now()
time = time.strftime("%m-%d_%H-%M")

filename = f"../data/Predictions/{last_year}_predictions_{time}_LSTM_GARCH.csv"
filepath = Path(filename)
filepath.parent.mkdir(parents=True, exist_ok = True)
df_test_whole.to_csv(filename)

df_test_whole.info()
print(df_test_whole.head())

/var/folders/3t/5vvh8l5x48s7tyvbx4b6v9xr0000gn/T/ipykernel_51555/3320645224.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test["Prediction"] = predictions


<class 'pandas.core.frame.DataFrame'>
Index: 1845482 entries, 3102998 to 5123797
Data columns (total 37 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   Unnamed: 0              int64  
 1   Quote_date              object 
 2   Expire_date             object 
 3   Price                   float64
 4   Underlying_last         float64
 5   Strike                  float64
 6   Ttl                     int64  
 7   Volatility              float64
 8   Volatility_GJR_GARCH    float64
 9   R                       float64
 10  Underlying_last-4       float64
 11  Strike-4                float64
 12  Ttl-4                   float64
 13  Volatility_GJR_GARCH-4  float64
 14  R-4                     float64
 15  Underlying_last-3       float64
 16  Strike-3                float64
 17  Ttl-3                   float64
 18  Volatility_GJR_GARCH-3  float64
 19  R-3                     float64
 20  Underlying_last-2       float64
 21  Strike-2                float6

In [16]:
# Run this if one month is very bad

if True:
    df_test_whole = pd.read_csv("../data/Predictions/2021_predictions_10-09_09-22_LSTM_GARCH.csv")
    df_test_whole.loc[(df_test_whole.loc[:, "Quote_date"] >= "2021-12-01 00:00:00") & (df_test_whole.loc[:, "Quote_date"] < "2022-01-01 00:00:00"), "Prediction"] = predictions

    from datetime import datetime
    time = datetime.now()
    time = time.strftime("%m-%d_%H-%M")

    filename = f"../data/Predictions/{last_year}_predictions_{time}_LSTM_GARCH.csv"
    filepath = Path(filename)
    filepath.parent.mkdir(parents=True, exist_ok = True)
    df_test_whole.to_csv(filename)

    df_test_whole.info()
    print(df_test_whole.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1845482 entries, 0 to 1845481
Data columns (total 40 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   Unnamed: 0.3            int64  
 1   Unnamed: 0.2            int64  
 2   Unnamed: 0.1            int64  
 3   Unnamed: 0              int64  
 4   Quote_date              object 
 5   Expire_date             object 
 6   Price                   float64
 7   Underlying_last         float64
 8   Strike                  float64
 9   Ttl                     int64  
 10  Volatility              float64
 11  Volatility_GJR_GARCH    float64
 12  R                       float64
 13  Underlying_last-4       float64
 14  Strike-4                float64
 15  Ttl-4                   float64
 16  Volatility_GJR_GARCH-4  float64
 17  R-4                     float64
 18  Underlying_last-3       float64
 19  Strike-3                float64
 20  Ttl-3                   float64
 21  Volatility_GJR_GARCH-3  float64